# 01 - Initial Scraping Notebook
## Instagram Competition Monitoring & Curation System

Notebook ini digunakan untuk pengumpulan data awal dari **Source Accounts** (60 hari terakhir) dan **Search Keywords** (Core: 30 hari, Discovery: 30 hari).
- Mendukung seleksi ulang sumber secara manual (centang akun/keyword yang ingin dijalankan).
- Menyimpan raw post ke System Log sebelum pemrosesan AI.
- Melakukan deduplikasi otomatis berdasarkan Post ID (shortcode).
- Menetapkan `global_checkpoint` pertama kali jika eksekusi berstatus **Success**.

### 1. Setup Lingkungan & Autentikasi Google Sheets
Memuat kode modul `compmon`, menginstal dependensi dari `requirements.txt`, dan melakukan autentikasi via `google.auth.default()` + `gspread.authorize()`.

In [ ]:
# @title 1. Muat Kode & Instal Dependensi Terpin
# Petunjuk: Pastikan folder 'compmon' dan 'requirements.txt' sudah diunggah ke runtime Colab,
# atau clone repositori jika menggunakan git.
import os, sys
sys.path.append(os.getcwd())

!pip install -q -r requirements.txt

from google.colab import auth, userdata
import google.auth
import gspread

# Autentikasi akun Google resmi Colab (Instruction 13)
auth.authenticate_user()
creds, _ = google.auth.default()
gc = gspread.authorize(creds)

# Membaca kredensial dari Colab Secrets
SPREADSHEET_ID = userdata.get('SPREADSHEET_ID')
APIFY_API_TOKEN = userdata.get('APIFY_API_TOKEN')
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

### 2. Parameter Eksekusi (Colab Form)
Pilih sumber yang ingin dijalankan. Biarkan kosong untuk menjalankan semua akun & keyword aktif.

In [ ]:
# @title Konfigurasi Seleksi Sumber (Opsional)
# @markdown Kosongkan untuk menjalankan semua sumber akun dan keyword aktif.
selected_accounts_str = "" # @param {type:"string"}
selected_keywords_str = "" # @param {type:"string"}

selected_accounts = [a.strip() for a in selected_accounts_str.split(',') if a.strip()] or None
selected_keywords = [k.strip() for k in selected_keywords_str.split(',') if k.strip()] or None

### 3. Eksekusi Initial Scraping & AI Processing

In [ ]:
from compmon.sheets_repo import SheetsRepository
from compmon.pipeline import CompetitionPipeline

repo = SheetsRepository(gc, SPREADSHEET_ID)
pipeline = CompetitionPipeline(
    sheets_repo=repo,
    apify_token=APIFY_API_TOKEN,
    gemini_api_key=GEMINI_API_KEY,
    notebook_name="INITIAL"
)

summary = pipeline.run_initial_scraping(
    selected_accounts=selected_accounts,
    selected_keywords=selected_keywords
)

print("=== Ringkasan Eksekusi Initial Scraping ===")
for key, val in summary.items():
    print(f"{key}: {val}")